# Decision trees and interpretations with Python

In [ ]:
# Loading data
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
bank_marketing = fetch_ucirepo(id=222) 
  
# data (as pandas dataframes) 
X = bank_marketing.data.features 
y = bank_marketing.data.targets 
  
# metadata 
print(bank_marketing.metadata) 
  
# variable information 
print(bank_marketing.variables) 


In [ ]:
# Consstruct dataframe from X,y
import pandas as pd
pd.set_option('display.max_columns', None)  # Show all columns in the dataframe
df = pd.DataFrame(X)
df['y'] = y
df.head()

# Data analysis and Visualization

In [ ]:
df['conversion'] = df['y'].map({'no': 0, 'yes': 1})
df.head()

In [ ]:
conversion_rate_df = pd.DataFrame(df.groupby('conversion')['y'].count() / df.shape[0] * 100.0)
conversion_rate_df

In [ ]:
conversion_rate_df.T

In [ ]:
conversion_rate_by_job_df = df.groupby(by='job')['conversion'].sum() / df.groupby(by='job')['conversion'].count() * 100.0
conversion_rate_by_job_df

In [ ]:
import plotly.express as px

plot_df = conversion_rate_by_job_df.reset_index()

fig = px.bar(
    plot_df,
    x='conversion',
    y='job',
    orientation='h',
    title='Conversion Rates by Job',
    color_discrete_sequence=['skyblue'],
    template='ggplot2'
)

fig.update_layout(
    xaxis_title='conversion rate (%)',
    yaxis_title='Job'
)
fig.update_xaxes(showgrid=True)

fig.show()

# Default rates by conversions

In [ ]:
default_by_conversion_df = pd.pivot_table(df, values='y', index='default', columns='conversion', aggfunc=len)
default_by_conversion_df

In [ ]:
import plotly.express as px
import plotly.colors as colors
plot_df = default_by_conversion_df.reset_index().melt(
    id_vars='default',
    var_name='conversion',
    value_name='count'
)

fig = px.pie(
    plot_df,
    names='default',
    values='count',
    facet_col='conversion',
    title='Default Rates by Conversion',
    template='ggplot2',
    color_discrete_sequence=colors.qualitative.Pastel
)

fig.update_traces(textinfo='percent+label')
fig.show()

# Bank balances by conversions

In [ ]:
plot_df = df[['conversion', 'balance']].copy()
plot_df['conversion'] = plot_df['conversion'].astype(str)

fig = px.box(
    plot_df,
    x='conversion',
    y='balance',
    points='outliers',
    title='Average Bank Balance Distributions by Conversion',
    template='ggplot2',
    color='conversion'
)

fig.update_layout(
    xaxis_title='Conversion',
    yaxis_title='Average Bank Balance',
    showlegend=False
)

fig.show()


In [ ]:
plot_df = df[['conversion', 'balance']].copy()
plot_df['conversion'] = plot_df['conversion'].astype(str)

fig = px.box(
    plot_df,
    x='conversion',
    y='balance',
    points=False,
    title='Average Bank Balance Distributions by Conversion',
    template='ggplot2',
    color='conversion'
)

fig.update_layout(
    xaxis_title='Conversion',
    yaxis_title='Average Bank Balance',
    showlegend=False
)

fig.show()


# Conversion rates by number of contacts

In [ ]:
conversions_by_num_contacts = (
	df.groupby(by='campaign')['conversion'].sum()
	/ df.groupby(by='campaign')['conversion'].count()
	* 100.0
)
conversions_by_num_contacts


In [ ]:
pd.DataFrame(conversions_by_num_contacts)

In [ ]:
plot_df = conversions_by_num_contacts.reset_index()
plot_df.columns = ['campaign', 'conversion_rate']

fig = px.bar(
    plot_df,
    x='campaign',
    y='conversion_rate',
    title='Conversion Rates by Number of Contacts',
    template='ggplot2',
    color_discrete_sequence=['skyblue']
)

fig.update_layout(
    xaxis_title='Number of Contacts',
    yaxis_title='Conversion Rate (%)'
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

fig.show()

# Encoding categorical variables for modeling

In [ ]:
df['month'].unique()

In [ ]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
df['month'] = df['month'].apply(lambda x: months.index(x)+1)
df['month'].unique()

In [ ]:
df.groupby(by='month')['conversion'].count()

In [ ]:
df['job'].unique()

In [ ]:
jobs_encoded_df = pd.get_dummies(df['job'])
jobs_encoded_df.columns = ['job_%s' % x for x in jobs_encoded_df.columns]
jobs_encoded_df


In [ ]:
df = pd.concat([df, jobs_encoded_df], axis=1)
df.head()

In [ ]:
# Encoding marital status
marital_encoded_df = pd.get_dummies(df['marital'])
marital_encoded_df.columns = ['marital_%s' % x for x in marital_encoded_df.columns]
marital_encoded_df.head()

In [ ]:
df = pd.concat([df, marital_encoded_df], axis=1)
df.head()

In [ ]:
# Enccoding Housing and Loan variables
df['housing'] = df['housing'].map({'no': 0, 'yes': 1})
df['loan'] = df['loan'].map({'no': 0, 'yes': 1})
df.head() 

# Building a decision tree model

In [ ]:
# Create feature list
features = [
    'age',
    'balance',
    'campaign',
    'previous',
    'housing',
] + list(jobs_encoded_df.columns) + list(marital_encoded_df.columns)

response_var = 'conversion'

In [ ]:
features

In [ ]:
from sklearn import tree

dt_model = tree.DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(df[features], df[response_var])


In [ ]:
dt_model.classes_

# Interpreting the decision tree model

In [ ]:
import graphviz
dot_data = tree.export_graphviz(
    dt_model,
    out_file=None,
    feature_names=features,
    class_names=['0', '1'],
    filled=True,
    rounded=True,
    special_characters=True
)
graph = graphviz.Source(dot_data)
graph.render("decision_tree")

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>text {font-size: 10px;}</style>"))

graph